# Downstream Training Demo

The official validity head in this repository is a PyTorch MLP in `code/downstream/train_validity.py`. This notebook trains a small CPU-only scikit-learn MLP on the same checked-in feature matrices so a beginner can see the training loop without needing a working Torch install.

The idea is the same: train on frozen transition-model features and predict `label_valid`.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

DATASET = ROOT / "data" / "validity_dataset"
CANDIDATES = DATASET / "candidates"
FEATURES = DATASET / "features"
HEADS = ROOT / "models" / "validity_heads"
RESULTS = ROOT / "results" / "analysis"

print(f"Repository root: {ROOT}")

Repository root: C:\Users\visha\OneDrive\Desktop\tokenizer-paper-workspace\planfm-validity


In [2]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score, average_precision_score, roc_auc_score, f1_score, confusion_matrix
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.exceptions import ConvergenceWarning
import warnings

family = "dd_xgb_wl_delta"
seed = 13


def load_split(split):
    data = np.load(FEATURES / family / f"seed_{seed}" / f"{split}.npz", allow_pickle=True)
    X = data["X"].astype(np.float32)
    y = data["y"].astype(int)
    return X, y, data

X_train, y_train, train_data = load_split("train")
X_val, y_val, val_data = load_split("validation")
X_test_i, y_test_i, test_i_data = load_split("test-interpolation")
X_test_e, y_test_e, test_e_data = load_split("test-extrapolation")

print(X_train.shape, np.bincount(y_train))

(1156, 55) [925 231]


In [3]:
# Standardize with train-set statistics only.
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_i_s = scaler.transform(X_test_i)
X_test_e_s = scaler.transform(X_test_e)

# The dataset is imbalanced, so I oversample positive training rows for this compact demo.
rng = np.random.default_rng(13)
pos_idx = np.flatnonzero(y_train == 1)
neg_idx = np.flatnonzero(y_train == 0)
extra_pos = rng.choice(pos_idx, size=len(neg_idx) - len(pos_idx), replace=True)
train_idx = np.concatenate([neg_idx, pos_idx, extra_pos])
rng.shuffle(train_idx)

X_bal = X_train_s[train_idx]
y_bal = y_train[train_idx]
print(f"Balanced demo training labels: {np.bincount(y_bal)} as [invalid, valid]")

Balanced demo training labels: [925 925] as [invalid, valid]


In [4]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore", ConvergenceWarning)
    clf = MLPClassifier(
        hidden_layer_sizes=(64, 32),
        activation="relu",
        alpha=1e-4,
        batch_size=64,
        learning_rate_init=1e-3,
        max_iter=120,
        early_stopping=True,
        validation_fraction=0.15,
        random_state=13,
    )
    clf.fit(X_bal, y_bal)

print(f"Demo MLP trained for {clf.n_iter_} iterations")

Demo MLP trained for 30 iterations


In [5]:
def evaluate(name, X, y):
    probs = clf.predict_proba(X)[:, 1]
    preds = (probs >= 0.5).astype(int)
    return {
        "split": name,
        "accuracy": accuracy_score(y, preds),
        "balanced_accuracy": balanced_accuracy_score(y, preds),
        "auroc": roc_auc_score(y, probs),
        "auprc": average_precision_score(y, probs),
        "f1": f1_score(y, preds),
        "tn_fp_fn_tp": confusion_matrix(y, preds, labels=[0, 1]).ravel().tolist(),
    }

metrics = pd.DataFrame([
    evaluate("validation", X_val_s, y_val),
    evaluate("test-interpolation", X_test_i_s, y_test_i),
    evaluate("test-extrapolation", X_test_e_s, y_test_e),
])
metrics

,split,accuracy,balanced_accuracy,auroc,auprc,f1,tn_fp_fn_tp
0,validation,0.969697,0.946970,0.947888,0.926530,0.923077,"[130, 2, 3, 30]"
1,test-interpolation,0.946154,0.930288,0.944989,0.916787,0.870370,"[199, 9, 5, 47]"
2,test-extrapolation,0.800733,0.503205,0.712377,0.430322,0.014493,"[1091, 1, 271, 2]"


In [6]:
official_command = "python -m code.downstream.train_validity --family dd_xgb_wl_delta --source_seed 13 --seed 13"
print("Official PyTorch training command:")
print(official_command)

Official PyTorch training command:
python -m code.downstream.train_validity --family dd_xgb_wl_delta --source_seed 13 --seed 13
